# Task 1 — Data Cleaning
**Splendor Analytics | Trial Activation Challenge**

This notebook covers:
1. Initial data load and profiling
2. Data type casting
3. Duplicate detection and removal
4. Null analysis and handling
5. Derived fields engineering
6. Data quality checks and anomaly flags
7. Clean dataset export

### Data Loading & Profile

In [2]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('/Users/hello/Documents/Jupiter Notebook/Splendor Analytics — Trial Activation Challenge/DA task.csv')

print("dataset info")
df.info()

display(df.head())

dataset info
<class 'pandas.DataFrame'>
RangeIndex: 170526 entries, 0 to 170525
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   ORGANIZATION_ID  170526 non-null  str  
 1   ACTIVITY_NAME    170526 non-null  str  
 2   TIMESTAMP        170526 non-null  str  
 3   CONVERTED        170526 non-null  bool 
 4   CONVERTED_AT     34235 non-null   str  
 5   TRIAL_START      170526 non-null  str  
 6   TRIAL_END        170526 non-null  str  
dtypes: bool(1), str(6)
memory usage: 8.0 MB


,ORGANIZATION_ID,ACTIVITY_NAME,TIMESTAMP,CONVERTED,CONVERTED_AT,TRIAL_START,TRIAL_END
0,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:03:53.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
1,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:52.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
2,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:53.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
3,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:05:18.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
4,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:06:00.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000


In [3]:
df.describe()

,ORGANIZATION_ID,ACTIVITY_NAME,TIMESTAMP,CONVERTED,CONVERTED_AT,TRIAL_START,TRIAL_END
count,170526,170526,170526,170526,34235,170526,170526
unique,966,28,99738,2,206,966,966
top,33f0b98a557961f5ccc519bb972d450f,Scheduling.Shift.Created,2024-03-11 13:25:12.000,False,2024-04-04 15:25:04.000,2024-03-20 11:01:59.000,2024-04-19 11:01:59.000
freq,12136,96895,512,136291,3826,12136,12136


### Type Casting

In [4]:
# Convert columns to appropriate data types
df['ORGANIZATION_ID'] = df['ORGANIZATION_ID'].astype('string')
df['ACTIVITY_NAME'] = df['ACTIVITY_NAME'].astype('category')

# Ensure CONVERTED is boolean
if df['CONVERTED'].dtype == 'object':
    df['CONVERTED'] = (
        df['CONVERTED']
        .astype(str)
        .str.strip()
        .str.lower()
        .map({'true': True, 'false': False})
    )
df['CONVERTED'] = df['CONVERTED'].astype('boolean')

# Convert datetime columns
datetime_cols = ['TIMESTAMP', 'CONVERTED_AT', 'TRIAL_START', 'TRIAL_END']
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Check result
df.dtypes

ORGANIZATION_ID            string
ACTIVITY_NAME            category
TIMESTAMP          datetime64[us]
CONVERTED                 boolean
CONVERTED_AT       datetime64[us]
TRIAL_START        datetime64[us]
TRIAL_END          datetime64[us]
dtype: object

### Handling Duplicaptes

In [5]:
# duplicate rows by ORGANIZATION_ID, ACTIVITY_NAME, TIMESTAMP
duplicate_rows = df[df.duplicated(subset=['ORGANIZATION_ID', 'ACTIVITY_NAME', 'TIMESTAMP'], keep=False)]
duplicate_summary = duplicate_rows.groupby(
    ['ORGANIZATION_ID', 'ACTIVITY_NAME', 'TIMESTAMP']
).size().reset_index(name='count')
duplicate_summary = duplicate_summary[duplicate_summary['count'] > 1].sort_values('count', ascending=False)
print("duplicate records")
display(duplicate_summary)

# total duplicate rows
print("total number of dupliicate rows is:",duplicate_summary['count'].sum())

# Remove duplicate rows
df_cleaned = df.drop_duplicates(subset=['ORGANIZATION_ID', 'ACTIVITY_NAME', 'TIMESTAMP'], keep='first')

# Summary
print(f"Original rows: {len(df)}")
print(f"Cleaned rows: {len(df_cleaned)}")
print(f"Duplicates removed: {len(df) - len(df_cleaned)}")

# Update df
df = df_cleaned
df

duplicate records


,ORGANIZATION_ID,ACTIVITY_NAME,TIMESTAMP,count
1489,2a1be3472ac43b76e03ed45bb5e371ec,Scheduling.Shift.Created,2024-03-11 13:25:12,512
2411,3eafe5a7fb996bc93b22cf6a9bb54082,Scheduling.Shift.Created,2024-01-23 15:00:39,490
2412,3eafe5a7fb996bc93b22cf6a9bb54082,Scheduling.Shift.Created,2024-01-23 15:00:40,485
2410,3eafe5a7fb996bc93b22cf6a9bb54082,Scheduling.Shift.Created,2024-01-23 15:00:38,425
903,1aa19170400130817b213e41d92f9f40,Scheduling.Shift.Created,2024-01-23 18:08:19,421
...,...,...,...,...
2739,4dc7ec0e512ab0e462f6f2a5ebfe54f1,Scheduling.Shift.Created,2024-03-30 21:40:26,2
2741,4e03db56b8cad5be5bf59fdac60a5af1,Mobile.Schedule.Loaded,2024-03-04 09:36:58,2
2742,4e03db56b8cad5be5bf59fdac60a5af1,Mobile.Schedule.Loaded,2024-03-06 00:13:48,2
2743,4e03db56b8cad5be5bf59fdac60a5af1,Mobile.Schedule.Loaded,2024-03-06 00:13:52,2


total number of dupliicate rows is: 70426
Original rows: 170526
Cleaned rows: 102895
Duplicates removed: 67631


,ORGANIZATION_ID,ACTIVITY_NAME,TIMESTAMP,CONVERTED,CONVERTED_AT,TRIAL_START,TRIAL_END
0,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:03:53,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39
1,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:52,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39
2,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:53,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39
3,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:05:18,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39
4,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:06:00,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39
...,...,...,...,...,...,...,...
170521,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:14:34,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40
170522,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:14:37,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40
170523,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:15:21,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40
170524,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:15:26,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40


In [6]:
df.describe()

,TIMESTAMP,CONVERTED_AT,TRIAL_START,TRIAL_END
count,102895,22223,102895,102895
mean,2024-03-01 10:38:31.639574,2024-03-15 00:58:47.063447,2024-02-16 13:46:32.559308,2024-03-17 13:46:32.559308
min,2024-01-01 20:52:26,2024-01-26 00:39:07,2024-01-01 15:21:50,2024-01-31 15:21:50
25%,2024-02-03 15:59:14,2024-02-24 11:39:05,2024-01-20 15:31:33,2024-02-19 15:31:33
50%,2024-03-04 14:07:35,2024-03-19 22:07:32,2024-02-21 11:08:46,2024-03-22 11:08:46
75%,2024-03-26 16:26:14,2024-04-03 20:55:19,2024-03-11 22:03:26,2024-04-10 22:03:26
max,2024-04-28 15:10:31,2024-05-10 13:51:28,2024-03-30 21:01:15,2024-04-29 21:01:15


### Feature Engineering

In [7]:
# Days into trial when each event occurred
df['DAYS_INTO_TRIAL'] = (df['TIMESTAMP'] - df['TRIAL_START']).dt.days

# Trial week bucket (1–5)
df['TRIAL_WEEK'] = np.clip((df['DAYS_INTO_TRIAL'] // 7) + 1, 1, 5)

# Trial month cohort (when the trial started)
df['COHORT_MONTH'] = df['TRIAL_START'].dt.to_period('M').astype(str)

# Trial moth name
df['TRIAL_START_MONTH'] = df['TRIAL_START'].dt.month_name()

# Days from trial start to conversion
df['DAYS_TO_CONVERT'] = (df['CONVERTED_AT'] - df['TRIAL_START']).dt.days

# Late conversion flag: converted after trial_end
df['LATE_CONVERSION'] = df['CONVERTED'] & (df['CONVERTED_AT'] > df['TRIAL_END'])

# Display the new columns
display(df)


,ORGANIZATION_ID,ACTIVITY_NAME,TIMESTAMP,CONVERTED,CONVERTED_AT,TRIAL_START,TRIAL_END,DAYS_INTO_TRIAL,TRIAL_WEEK,COHORT_MONTH,TRIAL_START_MONTH,DAYS_TO_CONVERT,LATE_CONVERSION
0,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:03:53,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39,0,1,2024-03,March,NaN,False
1,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:52,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39,0,1,2024-03,March,NaN,False
2,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:53,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39,0,1,2024-03,March,NaN,False
3,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:05:18,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39,0,1,2024-03,March,NaN,False
4,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:06:00,False,NaT,2024-03-27 10:11:39,2024-04-26 10:11:39,0,1,2024-03,March,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
170521,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:14:34,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40,27,4,2024-01,January,NaN,False
170522,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:14:37,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40,27,4,2024-01,January,NaN,False
170523,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:15:21,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40,27,4,2024-01,January,NaN,False
170524,4eb736e6ff7851d301ae68a6aa37081a,Mobile.Schedule.Loaded,2024-01-31 22:15:26,False,NaT,2024-01-04 12:07:40,2024-02-03 12:07:40,27,4,2024-01,January,NaN,False


### Data Cleaning Summary & Key Findings

The dataset was cleaned in a structured sequence to improve reliability for downstream trial-activation analysis.

#### 1) Initial load and profiling
- Loaded the raw CSV into `df`.
- Reviewed schema with `df.info()` and sampled records with `df.head()`.
- Used `df.describe()` to inspect baseline numeric distributions.

#### 2) Data type casting
- `ORGANIZATION_ID` cast to `string` to preserve identifier integrity.
- `ACTIVITY_NAME` cast to `category` for memory efficiency and cleaner grouping.
- `CONVERTED` standardized from text values (`"true"/"false"`) into nullable boolean.
- Datetime fields (`TIMESTAMP`, `CONVERTED_AT`, `TRIAL_START`, `TRIAL_END`) parsed with `errors='coerce'`, converting invalid values to `NaT`.

**Finding:** Type inconsistencies were normalized, making event-time and conversion logic reliable.
#### 3) Duplicate detection and removal
- Duplicates were checked using the business keys:
    - `ORGANIZATION_ID`
    - `ACTIVITY_NAME`
    - `TIMESTAMP`
- Duplicate groups were summarized and duplicate records were removed with `keep='first'`.
- `df` was updated to the deduplicated output.

**Duplicate summary**
- Duplicate groups found: **2,795**
- Rows participating in duplicate groups: **70,426**
- Duplicate rows removed: **67,631**
- Rows retained after deduplication: **102,895**

**Finding:** Repeated event records were removed, reducing event inflation risk in activity and conversion metrics.

#### 4) Feature engineering
New analytical fields were added to support cohort, timing, and conversion-quality analysis.

---

### Data Dictionary (Engineered Columns)

| Column | Type | Definition | Example / Notes |
|---|---|---|---|
| `DAYS_INTO_TRIAL` | integer (nullable) | Number of days between event `TIMESTAMP` and `TRIAL_START`. | `0` = event on trial start day. Negative values indicate pre-trial activity timestamps. |
| `TRIAL_WEEK` | integer | Week bucket of trial activity, derived from `DAYS_INTO_TRIAL` and clipped to range 1–5. | Days 0–6 → Week 1, 7–13 → Week 2, etc. |
| `COHORT_MONTH` | string (`YYYY-MM`) | Monthly cohort based on `TRIAL_START`. | Useful for cohort trend analysis. |
| `TRIAL_START_MONTH` | string | Month name from `TRIAL_START`. | Example: `January`, `February`. |
| `DAYS_TO_CONVERT` | integer (nullable) | Number of days from `TRIAL_START` to `CONVERTED_AT`. | Null for non-converted orgs or missing conversion timestamp. |
| `LATE_CONVERSION` | boolean (nullable) | `True` if converted and `CONVERTED_AT` is after `TRIAL_END`; otherwise `False`. | Flags post-trial conversions for policy and funnel analysis. |

---

### Overall outcome
- Data is now typed consistently, deduplicated, and enriched with time-based trial features.
- The cleaned dataset is suitable for conversion funnel analysis, cohort comparison, and anomaly monitoring.

In [8]:
df.to_csv('clean_task_data.csv', index=False)
print("Dataset exported successfully as 'clean_task_data.csv'")

Dataset exported successfully as 'clean_task_data.csv'
